# 🚗 FocalComm: Hard Instance-Aware Multi-Agent Perception (WACV 2026)
### 📘 Pipeline Huấn Luyện & Đánh Giá Toàn Diện Trên Kaggle GPU (T4 / P100)

Notebook này kết nối trực tiếp với mã nguồn chuẩn của dự án từ GitHub: [tranvuquanghuy12/FocalComm](https://github.com/tranvuquanghuy12/FocalComm.git).

**Nội dung thực thi gồm 3 phần độc lập, linh hoạt:**
1. **Baseline Model (Fixed):** Huấn luyện mô hình cơ sở đã sửa triệt để lỗi đảo nhãn và hồi quy góc quay yaw.
2. **FocalComm Official Training:** Huấn luyện mô hình cộng tác đa phương tiện với module HIM (Hard Instance Mining) và QAFF.
3. **Instant Official Evaluation:** Tải trọng số pretrained chính thức của bài báo (WACV 2026) để nghiệm thu ngay bảng kết quả **56.1% mAP**.

## 1. Kiểm tra cấu hình phần cứng & GPU trên Kaggle
Xác nhận GPU (T4 / P100), PyTorch và phiên bản CUDA khả dụng.

In [ ]:
import os
import sys
import torch

print(f"Python Version: {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️ CẢNH BÁO: CUDA chưa bật! Vui lòng chọn Accelerator = GPU trong cài đặt bên phải Kaggle.")

!nvidia-smi


## 2. Clone Repository FocalComm từ GitHub
Lấy mã nguồn mới nhất từ GitHub repository.

In [ ]:
# Xóa thư mục cũ nếu có để đảm bảo mã nguồn luôn cập nhật nhất
!rm -rf /kaggle/working/FocalComm
!git clone https://github.com/tranvuquanghuy12/FocalComm.git /kaggle/working/FocalComm

%cd /kaggle/working/FocalComm
!git log -n 1 --oneline
!ls -la


## 3. Cài đặt các thư viện cần thiết & Biên dịch CUDA/Cython Extensions
Cài đặt các gói phụ thuộc và build các module xử lý 3D hình học (box_overlaps, iou3d_nms).

In [ ]:
# 1. Cài đặt các packages cần thiết
!pip install -q easydict numba open3d cython tensorboardX shapely==2.0.0
!pip install -q pyquaternion einops timm icecream python-lzf termcolor efficientnet_pytorch==0.7.0 huggingface_hub
!pip install -q git+https://github.com/klintan/pypcd.git

# 2. Cài đặt spconv phù hợp với môi trường Kaggle
!pip install -q spconv-cu118 || pip install -q spconv-cu120 || pip install -q spconv-cu117

# 3. Biên dịch CUDA & Cython Extensions
%cd /kaggle/working/FocalComm/focalcomm/utils
!python setup.py build_ext --inplace

%cd /kaggle/working/FocalComm/focalcomm/pcdet_utils
!python setup.py build_ext --inplace

%cd /kaggle/working/FocalComm
print("✅ Môi trường và CUDA Extensions đã sẵn sàng!")


## 4. Thiết lập Bộ Dữ Liệu V2X-Real
Liên kết thư mục dữ liệu V2X-Real từ Kaggle input hoặc tải dữ liệu mẫu.

In [ ]:
import os
import shutil

DATASET_ROOT = "/kaggle/working/dataset/V2XReal"
os.makedirs(DATASET_ROOT, exist_ok=True)

# Tự động phát hiện dữ liệu nếu bạn đã add dataset trên Kaggle
kaggle_input_dirs = [d for d in os.listdir("/kaggle/input") if "v2x" in d.lower() or "focalcomm" in d.lower()] if os.path.exists("/kaggle/input") else []
if kaggle_input_dirs:
    src = os.path.join("/kaggle/input", kaggle_input_dirs[0])
    print(f"Phát hiện dataset trong Kaggle Input: {src}")
    # Tạo symlink để đọc nhanh mà không tốn dung lượng ổ đĩa
    if not os.path.exists(os.path.join(DATASET_ROOT, "train")) and os.path.exists(os.path.join(src, "train")):
        os.symlink(os.path.join(src, "train"), os.path.join(DATASET_ROOT, "train"))
    if not os.path.exists(os.path.join(DATASET_ROOT, "val")) and os.path.exists(os.path.join(src, "val")):
        os.symlink(os.path.join(src, "val"), os.path.join(DATASET_ROOT, "val"))
    print(f"Liên kết dữ liệu thành công vào: {DATASET_ROOT}")
else:
    print(f"Dataset sẽ được đọc từ thư mục: {DATASET_ROOT}")
    print("Nếu bạn có file zip dữ liệu, hãy giải nén vào thư mục này.")


## 5. LỰA CHỌN 1: Huấn Luyện Mô Hình Baseline (Đã Sửa Triệt Để Lỗi)
Chạy script `focalcomm/tools/train_baseline.py` trực tiếp từ repo. Script này tích hợp:
* Chuẩn hóa 3 lớp đồng bộ hoàn toàn (Car: 0, Pedestrian: 1, Truck: 2).
* Hồi quy góc quay yaw 8 kênh vector (sin, cos) giải mã qua atan2 (loại bỏ hoàn toàn lỗi asin).
* Kích thước Anchor 3D Priors cho từng đối tượng.
* Đánh giá chuẩn AP@0.3, AP@0.5, AP@0.7 đa ngưỡng.

In [ ]:
# Thêm cờ --subset nếu bạn muốn kiểm tra chạy thử nhanh (Dry-run)
!python focalcomm/tools/train_baseline.py \
    --data_dir /kaggle/working/dataset/V2XReal \
    --epochs 30 \
    --batch_size 4 \
    --save_dir checkpoints/baseline_v2xreal


## 6. LỰA CHỌN 2: Huấn Luyện FocalComm Hoàn Chỉnh (HIM + QAFF Intermediate Fusion)
Chạy huấn luyện mô hình FocalComm v3 chính thức với cấu hình YAML tối ưu cho Kaggle GPU.

In [ ]:
# Huấn luyện FocalComm với cấu hình Kaggle đã được tinh chỉnh bộ nhớ VRAM
!python focalcomm/tools/train_focalcomm.py \
    --hypes_yaml focalcomm/hypes_yaml/v2xreal/focalcommv3_kaggle.yaml


## 7. LỰA CHỌN 3: Nghiệm Thu Tức Thì Bằng Checkpoint Chính Thức từ Hugging Face
Nếu bạn cần số liệu thực nghiệm ngay lập tức để làm slide/báo cáo, ô này sẽ tải trọng số chính thức của bài báo FocalComm (WACV 2026) và chạy kiểm thử để tái hiện con số **56.1% mAP**.

In [ ]:
from huggingface_hub import snapshot_download
import os

ckpt_dir = "checkpoints/focalcomm_v2xreal"
os.makedirs(ckpt_dir, exist_ok=True)

print("Đang tải checkpoint chính thức từ Hugging Face (scdrand23/FocalComm)...")
try:
    snapshot_download(
        repo_id="scdrand23/FocalComm",
        local_dir="checkpoints",
        allow_patterns=["*focalcomm_v2xreal*", "*net_epoch_best.pth*"]
    )
    print("✅ Đã tải checkpoint thành công!")
except Exception as e:
    print(f"Thông báo: {e}")

# Chạy đánh giá trên tập kiểm thử (Inference)
!python focalcomm/tools/inference.py \
    --model_dir checkpoints/focalcomm_v2xreal \
    --fusion_method intermediate


## 8. Đóng gói và Tải Checkpoints / Kết Quả Thực Nghiệm
Nén kết quả thành file `.tar.gz` để tải về máy cục bộ.

In [ ]:
output_tar = "/kaggle/working/focalcomm_results.tar.gz"
!tar -czvf {output_tar} checkpoints/
print(f"✅ Đã tạo file nén tại: {output_tar}")
print("Bạn có thể tải file này trực tiếp từ tab Output bên phải màn hình Kaggle.")
